# PsychScanner Parser Modules — Tutorial

This tutorial walks you through how parser classes are organized in the `psychscanner` package after the recent reorganization.

## Layout at a glance

| Module | Contents | When to use |
|--------|----------|-------------|
| `psychscanner.parsers`                                | **Top-level namespace.** Re-exports every bundled parser + `list_parsers()` / `get_parser()` registry helpers. | New code — import from here. |
| `psychscanner.datasets.prompts.parser_tasks`          | Task/paradigm-specific parsers (vividness + reality-monitoring). | When you want to inspect or extend a specific paradigm's parsers. |
| `psychscanner.datasets.prompts.parser_general`        | Paradigm-agnostic parsers (Likert, generic response+rating, word classification, readiness, `TaskResponse`). | When you need a generic shape. |
| `psychscanner.datasets.prompts.parser`                | **Backward-compat shim.** Every name previously importable here still works. | Existing code — keep working unchanged. |
| `psychscanner.datasets.prompts.parser_extra`          | **Backward-compat shim.** Every name previously importable here still works. | Existing code — keep working unchanged. |

**Important:** all four import paths return the *same class objects* — there is no duplication. The shims just import from the themed modules.

## 1. Setup

In [ ]:
import psychscanner as psy
from psychscanner.parsers import list_parsers, get_parser, PARSER_REGISTRY

print('psychscanner version:', psy.__version__)
print('Total parsers in registry:', len(PARSER_REGISTRY))

## 2. Discover what's available

`list_parsers()` returns every bundled parser class name, sorted.

In [ ]:
names = list_parsers()
print(f'{len(names)} bundled parsers:')
for n in names:
    print(f'  - {n}')

## 3. Look at the themed split

Inspect each themed module to see which parsers belong to which group.

In [ ]:
import inspect
from pydantic import BaseModel
from psychscanner.datasets.prompts import parser_tasks, parser_general

def native_parsers(mod):
    """Classes actually defined in `mod` (not just re-imported)."""
    return sorted(
        name for name, obj in inspect.getmembers(mod, inspect.isclass)
        if issubclass(obj, BaseModel) and obj is not BaseModel
        and obj.__module__ == mod.__name__
    )

print('parser_tasks (paradigm-specific):')
for n in native_parsers(parser_tasks): print(f'  - {n}')
print()
print('parser_general (paradigm-agnostic):')
for n in native_parsers(parser_general): print(f'  - {n}')

## 4. Three ways to import the same parser

Pick whichever feels most readable in your code — they all return the **same class object**.

In [ ]:
# Way A — top-level namespace (recommended for new code)
from psychscanner.parsers import DefaultLiteralVivid15 as A

# Way B — themed module (when grouping with related parsers)
from psychscanner.datasets.prompts.parser_tasks import DefaultLiteralVivid15 as B

# Way C — backward-compat shim (existing code keeps working)
from psychscanner.datasets.prompts.parser import DefaultLiteralVivid15 as C

print('All three are the same class object:', A is B is C)
print('Class:', A.__name__)
print('Defined in:', A.__module__)

## 5. Look up by name (string lookup)

When the parser name comes from a config file or a task JSON, use `get_parser()`.

In [ ]:
# String-based lookup
parser_cls = get_parser('DefaultLiteralAgree')
print('Resolved:', parser_cls.__name__)
print('Defined in:', parser_cls.__module__)

# Misspelled? Get a clear error with the available options listed.
try:
    get_parser('NoSuchParser')
except KeyError as e:
    msg = str(e)
    print('Error message starts with:')
    print(' ', msg[:120], '...')

## 6. Inspect a parser's schema

Every parser is a Pydantic `BaseModel`, so you can introspect its JSON schema, fields, and field descriptions.

In [ ]:
import json
from psychscanner.parsers import DefaultLiteralVivid15

print('Field names:    ', list(DefaultLiteralVivid15.model_fields.keys()))
print('Required fields:', list(DefaultLiteralVivid15.model_fields.keys()))
print()
print('JSON schema:')
print(json.dumps(DefaultLiteralVivid15.model_json_schema(), indent=2)[:600], '...')

## 7. Use a parser in an `ExpCard`

Pass any parser class directly via the `parser=` kwarg.

In [ ]:
import shutil
from pathlib import Path
from psychscanner import ExpCard, ExpCardInit, ScannerModel
from psychscanner.parsers import DefaultLiteralVivid15

PROJECT_DIR = Path.cwd() / '_parser_modules_tutorial_runs'
if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
PROJECT_DIR.mkdir()

task_file = Path.cwd() / 'tasks' / 'example_survey.json'

card = ExpCardInit()
card.proj_dir      = PROJECT_DIR
card.projectname   = 'direct_class'
card.model         = 'mock-chat-model'   # mock for a runnable demo
card.family        = 'mock-llm'
card.task_file     = task_file
card.parser        = '0'                 # mock-llm only supports no-parser mode
card.cogtype       = 'no'
card.nsim          = 1
card.tunnel_status = '0'

exp = ExpCard(card)
print('Card resolved parser:', exp.parser)
print('Output dir:          ', exp.data_root_dir)

## 8. Resolve a parser by name from a task JSON (`parser="1"`)

When `parser='1'`, `psychscanner` reads the parser class name from the task JSON's top-level `"parser"` field and resolves it through the registry. The example survey ships with `"parser": "DefaultLiteralAgree"`.

In [ ]:
import json
task_json = json.loads(task_file.read_text())
print('Task JSON declares parser:', task_json['parser'])

card2 = ExpCardInit()
card2.proj_dir      = PROJECT_DIR
card2.projectname   = 'json_lookup'
card2.task_file     = task_file
card2.parser        = '1'                # ← resolve from task JSON
card2.cogtype       = 'no'
card2.nsim          = 1
card2.tunnel_status = '0'

exp2 = ExpCard(card2)
print('Card resolved parser:', exp2.parser.__name__)
print('Defined in:          ', exp2.parser.__module__)

## 9. What if the JSON references an unknown parser?

You get a clear error message listing every available parser — no more cryptic `NameError` from an `eval()` call.

In [ ]:
import tempfile

bad_task = {
    'tasktype': 'survey', 'taskname': 'bad', 'instructions': {'definition': ['x']},
    'contexts': ['A'], 'contexts_id': ['A'], 'context_present': True,
    'items': {'A_1': [{'trcode': 'A_1', 'stimulus': 'x'}]},
    'parser': 'TotallyMadeUpClass',
    'chain_type': 'item',
}
tmp_path = Path(tempfile.mkdtemp()) / 'bad.json'
tmp_path.write_text(json.dumps(bad_task))

card_bad = ExpCardInit()
card_bad.task_file     = tmp_path
card_bad.parser        = '1'
card_bad.cogtype       = 'no'
card_bad.nsim          = 1
card_bad.tunnel_status = '0'
card_bad.proj_dir      = PROJECT_DIR / 'bad_parser_attempt'

try:
    ExpCard(card_bad)
except ValueError as e:
    print('Caught ValueError:')
    print(' ', str(e)[:160], '...')

## 10. Define your own parser

Any `pydantic.BaseModel` subclass works. Pass it directly via `parser=`.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class YesNoConfidence(BaseModel):
    """Answer yes/no and rate your confidence on a 1–5 scale."""
    answer:     Literal['yes', 'no']    = Field(..., description='Your yes/no answer')
    confidence: Literal[1, 2, 3, 4, 5]  = Field(..., description='1=guess, 5=certain')

card_custom = ExpCardInit()
card_custom.task_file     = task_file
card_custom.parser        = YesNoConfidence    # custom Pydantic class
card_custom.cogtype       = 'no'
card_custom.nsim          = 1
card_custom.tunnel_status = '0'
card_custom.proj_dir      = PROJECT_DIR / 'custom_parser'
card_custom.model         = 'mock-chat-model'
card_custom.family        = 'mock-llm'

# We can't actually run YesNoConfidence with mock-llm (mock doesn't do structured
# output), but ExpCard still validates and stores the parser.
exp_c = ExpCard(card_custom)
print('Resolved parser :', exp_c.parser.__name__)
print('Schema fields  :', list(exp_c.parser.model_fields.keys()))

## 11. Mental model — when to use which import path

```python
# Best for new code: top-level namespace, short and discoverable
from psychscanner.parsers import DefaultLiteralVivid15

# When you want to pick from a paradigm group:
from psychscanner.datasets.prompts.parser_tasks   import DefaultLiteralVivid15
from psychscanner.datasets.prompts.parser_general import DefaultLiteralAgree

# Pre-existing imports continue to work unchanged:
from psychscanner.datasets.prompts.parser       import DefaultLiteralVivid15
from psychscanner.datasets.prompts.parser_extra import DefaultLiteralAgree
```

Pick whichever path makes the import line read most naturally — the package treats them as equivalent.

## Summary

1. **`parser_tasks`** = vividness + reality-monitoring (paradigm-specific).
2. **`parser_general`** = Likert, response/rating, word-classification, readiness, `TaskResponse` Union (paradigm-agnostic).
3. **`parser` and `parser_extra`** = backward-compat shims; nothing breaks.
4. **`psychscanner.parsers`** = single registry + `list_parsers()` + `get_parser(name)`.
5. `parser="1"` reads the class name from the task JSON's `"parser"` key, looks it up in the registry, and uses it. Misspellings produce a clear error with all available names listed.

All previously-shipped parser classes are unchanged — only file layout was streamlined.